In [1]:
%pip install -U scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   ------------------------------- -------- 6.6/8.2 MB 27.0 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 21.2 MB/s  0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.8.0
    Uninstalling scikit-learn-1.8.0:
      Successfully uninstalled scikit-learn-1.8.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Importing necessary libraries
import pandas as pd
import boto3
import json
import gzip
from datetime import datetime
from io import StringIO, BytesIO
import requests
import numpy as np
from sklearn.neighbors import NearestNeighbors

In [ ]:
def fetch_annual_traffic_counts():
    """
    Fetches annual traffic counts from the NCDOT API for specified location IDs.

    Args: None, the IDs are pre-specified in the function.
    
    Returns:
        ncdot_df: A DataFrame containing traffic data (NCDOT average annual traffic counts) for the specified locations.
    """

    url = "https://services.arcgis.com/NuWFvHYDMVmmxMeM/ArcGIS/rest/services/NCDOT_AADT_Stations/FeatureServer/0/query"
    #Locations we will be focusing on
    location_ids = [
        "0600000063",
        "0600000305",
        "0600000145",
        "0600000829",
        "0600000107",
        "0600000108",
        "0600000414",
        "0600000412",
        "0600000044",
        "0600000065",
        "0600001182"
    ]
    # Construct the where clause for the API request based on the location IDs
    where = "LocationID IN ({})".format(",".join(f"'{x}'" for x in location_ids))
    params = {
        "where": where,
        "outFields": "LocationID,COUNTY,ROUTE,LOCATION,AADT_2022",
        "returnGeometry": "true",
        "outSR": '{"wkid":4326}',
        "f": "json"
    }
    # Send the API request to fetch the traffic data for the specified location IDs
    response = requests.get(url, params=params)
    response.raise_for_status()
    #
    api_data = response.json()
    #checks that the coordinates were updated correctly
    print(api_data["features"][0])

    ncdot_df = pd.json_normalize(api_data["features"])

    #renaming columns
    ncdot_df = ncdot_df.rename(columns={
        "attributes.LocationID": "LocationID",
        "attributes.COUNTY": "County",
        "attributes.ROUTE": "Route",
        "attributes.LOCATION": "Location",
        "attributes.AADT_2022": "AADT_2022",
        "geometry.x": "longitude",
        "geometry.y": "latitude"
    })

    return ncdot_df


{'attributes': {'LocationID': '0600000044', 'COUNTY': 'MECKLENBURG', 'ROUTE': 'US 29/NC 49', 'LOCATION': 'EAST OF STETSON DR', 'AADT_2022': '29500'}, 'geometry': {'x': -80.75957420178952, 'y': 35.28798544117831}}
    LocationID       County                     Route  \
0   0600000044  MECKLENBURG               US 29/NC 49   
1   0600000063  MECKLENBURG                     NC 49   
2   0600000065  MECKLENBURG                     US 29   
3   0600000107  MECKLENBURG                     NC 49   
4   0600000108  MECKLENBURG                     NC 49   
5   0600000145  MECKLENBURG                     NC 49   
6   0600000305  MECKLENBURG                     US 29   
7   0600000412  MECKLENBURG    NC 24 (WT HARRIS BLVD)   
8   0600000414  MECKLENBURG  NC 24 (E WT HARRIS BLVD)   
9   0600000829  MECKLENBURG                     NC 49   
10  0600001182  MECKLENBURG                   SR 2833   

                            Location AADT_2022  longitude   latitude  
0                 EAST OF STETS

In [ ]:
def save_traffic_counts_to_s3(ncdot_df):
    """
    Saves the traffic counts DataFrame to an S3 bucket in CSV format with the current timestamp.

    Args:
        ncdot_df: A DataFrame containing traffic data (NCDOT average annual traffic counts).
    
    Returns:
        None
    """
    # Include date and time
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S-%f")
    file_key = f"ncdot/traffic/aadt_stations_{timestamp}.csv"

    # Compress the JSON data
    csv_string = ncdot_df.to_csv(index=False)

    # Upload to S3
    s3_client.put_object(
        Bucket=bucket,
        Key=file_key,
        Body=csv_string.encode("utf-8"),
        ContentType="text/csv",
    )

    print("Saved file:")
    print(f"s3://{bucket}/{file_key}")

Saved file:
s3://s3-3601-final-project/tomtom/traffic/aadt_stations_2026-09-27_15-13-28-302554.csv


In [ ]:
# Setting up S3 client and defining bucket and prefix for traffic data
s3_client = boto3.client("s3")

bucket = "s3-3601-final-project"
prefix = "ncdot/traffic/"

ncdot_df = fetch_annual_traffic_counts()
save_traffic_counts_to_s3(ncdot_df)